In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-04-fingerprint-attribution.py"

# 把指紋位元對回對氯苯甲醚的結構

以實測的 ESOL 水溶解度訓練模型。Colab 首次執行會下載資料。以下分析在獨立的 Python 程序中執行；試做練習時，可把程式接在分析格末端，再重跑分析格與顯示格。

In [ ]:
%%writefile "{script}"
from collections import defaultdict
from pathlib import Path
from io import BytesIO
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Delaney ESOL，實測 log10(S / (mol/L))。DOI 10.1021/ci034243x
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
observed = defaultdict(list)
for smi, y, w in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    if w > 0 and np.isfinite(y):
        observed[Chem.MolToSmiles(Chem.MolFromSmiles(smi))].append(float(y))
train, test = train_test_split(sorted(observed), test_size=0.2, random_state=42)
def labels(smiles):
    return np.array([np.mean(observed[s]) for s in smiles])

target = "COc1ccc(Cl)cc1"  # 對氯苯甲醚
assert target in test
mol = Chem.MolFromSmiles(target)
print("train / test:", len(train), len(test), "| observed log S:", labels([target])[0])

def structure(atom_colors, bond_colors):
    drawing = rdMolDraw2D.MolDraw2DCairo(600, 380)
    drawing.drawOptions().addAtomIndices = True
    drawing.drawOptions().useBWAtomPalette()
    drawing.DrawMolecule(mol, highlightAtoms=list(atom_colors), highlightBonds=list(bond_colors),
                         highlightAtomColors=atom_colors, highlightBondColors=bond_colors)
    drawing.FinishDrawing()
    return Image.open(BytesIO(drawing.GetDrawingText())).convert("RGB")

out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import Ridge

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
def matrix(smiles):
    return np.stack([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in smiles])

X_train = matrix(train)
model = Ridge(alpha=2.0).fit(X_train, labels(train))
print("test RMSE:", root_mean_squared_error(labels(test), model.predict(matrix(test))))
info = rdFingerprintGenerator.AdditionalOutput()
info.CollectBitInfoMap()
x = gen.GetFingerprintAsNumPy(mol, additionalOutput=info)
bit_info = info.GetBitInfoMap()
# 線性模型、以訓練集作背景、缺席特徵以邊際期望補入。
phi = (x - X_train.mean(axis=0)) * model.coef_
base = model.predict(X_train).mean()
full = model.predict(x[None, :])[0]
assert np.isclose(base + phi.sum(), full)

def environment(center, radius):
    bonds = list(Chem.FindAtomEnvironmentOfRadiusN(mol, radius, center))
    atoms = {center}
    for idx in bonds:
        bond = mol.GetBondWithIdx(idx)
        atoms.update([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
    return atoms, bonds

# 列出每個位元的全部來源，再逐一遮去包含指定原子的環境。
occurrences = {bit: [environment(*pair) for pair in entries] for bit, entries in bit_info.items()}
def mask_atom(atom):
    masked = x.copy()
    for bit, sources in occurrences.items():
        if all(atom in atoms for atoms, bonds in sources):
            masked[bit] = 0
    return masked

weights = np.array([full - model.predict(mask_atom(i)[None, :])[0] for i in range(mol.GetNumAtoms())])
ranked = sorted(bit_info, key=lambda bit: -abs(phi[bit]))
print("base / predicted:", base, full)
for bit in ranked[:8]:
    print("bit / SHAP / occurrences:", bit, round(phi[bit], 3), bit_info[bit])
print("atom mask differences:", weights.round(3))
# 選一個在該分子出現多次的位元，看對稱位置。
bit = next(b for b in ranked if len(bit_info[b]) > 1 and bit_info[b][0][1] > 0)
orange = (1.0, 0.72, 0.34)
atoms = {i: orange for ids, bonds in occurrences[bit] for i in ids}
bonds = {j: orange for ids, edges in occurrences[bit] for j in edges}
scale = max(abs(weights))
def shade(value):
    return tuple(plt.get_cmap("RdBu_r")((1 + value / scale) / 2)[:3])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), layout="constrained")
axes[0].imshow(structure(atoms, bonds))
axes[0].set_title(f"Bit {bit} | SHAP {phi[bit]:+.2f} log S")
axes[1].imshow(structure({i: shade(v) for i, v in enumerate(weights)}, {}))
axes[1].set_title("Mask environments at one atom")
for ax in axes:
    ax.axis("off")
mappable = plt.cm.ScalarMappable(norm=plt.Normalize(-scale, scale), cmap="RdBu_r")
fig.colorbar(mappable, ax=axes[1], shrink=.74, label="Full minus masked (log S)")
fig.savefig(out / "11-04-fingerprint-attribution.png", dpi=170)
print("illustrated bit:", bit, bit_info[bit])


In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-04-fingerprint-attribution.png")))